![image_1779878561360.png](./image_1779878561360.png "image_1779878561360.png")

![image_1779878588075.png](./image_1779878588075.png "image_1779878588075.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import *

# Initialize Spark session
spark = SparkSession.builder.appName("TransactionsDF").getOrCreate()

# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("trans_date", StringType(), True)  # can convert later to DateType
])

# Data
data = [
    (1, "US", "approved", 1000, "2023-01-01"),
    (2, "US", "declined", 2000, "2023-01-02"),
    (3, "US", "approved", 1500, "2023-02-03"),
    (4, "DE", "approved", 2000, "2023-01-04")
]

# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

# Convert string to date (optional but recommended)
from pyspark.sql.functions import to_date
# df = (
#     df
#     .select(
#         date_format(df.trans_date, 'yyyy-MM').alias("month"),
#         df.country,
#         df.state,
#         df.amount
#     )
# )
# df_result=(
#     df.select(
#         df.month,
#         df.country,
#         count(when(df.state == 'approved', 1)).alias("approved_count"),
#         count(df.state).alias("trans_count"),
#         sum(df.amount).alias("trans_total_amount"),
#         sum(when(df.state == 'approved',df.amount).alias("approved_total_amount")
#             .gropupBy(df.month, df.country)      
# )
#     )

df_result=(
    df.groupBy(date_format("trans_date", "yyyy-MM").alias("month"), df.country)
    .agg(
        count(df.state).alias("trans_count"),
        count(when(df.state == 'approved', 1)).alias("approved_count"),
        sum(df.amount).alias("trans_total_amount"),
        sum(when(df.state == 'approved', df.amount).otherwise(0)).alias("approved_total_amount")
      )

)
# Show DataFrame
df_result.show()